In [ ]:
%%capture
!pip install aixplain

import os
os.environ["AIXPLAIN_API_KEY"] = "YOUR-API-KEY"

from aixplain.factories import AgentFactory, ToolFactory, ModelFactory

In [ ]:
# Get required tools
# Note: Get these IDs from studio.aixplain.com after connecting your Perplexity and Google accounts
# Navigate to: studio.aixplain.com -> Integrations -> Connect Perplexity and Google Sheets
# Then go to Tools section to copy the tool IDs

perplexity_tool = ToolFactory.get("YOUR-PERPLEXITY-TOOL-ID")  # Replace with your Perplexity tool ID
google_sheets_tool = ToolFactory.get("YOUR-GOOGLE-SHEETS-TOOL-ID")  # Replace with your Google Sheets tool ID

In [ ]:
# Get Claude Code model (or your preferred LLM)
claude_code = ModelFactory.get("claude-code")  # Replace with actual Claude Code model ID

In [ ]:
# Create Content Strategy Research Agent
research_agent = AgentFactory.create(
    name="Perplexity Research Agent",
    description="A content strategy research agent that identifies high-value content opportunities for B2B AI companies",
    llm_id=claude_code.id,
    instructions="""
    You are a content strategy research agent specializing in identifying high-value content opportunities for B2B AI companies.

    When given a company domain and topic focus, you will use Perplexity tool to:
    1. Research the current content landscape around the specified topics
    2. Identify trending questions, pain points, and knowledge gaps in the target audience
    3. Analyze what competitors are NOT covering well
    4. Find emerging trends and discussions in relevant communities (Reddit, LinkedIn, forums)
    5. Assess search intent and content formats that are performing well
    6. Prioritize opportunities based on: search demand, competition level, and alignment with the company's positioning

    IMPORTANT: You must identify at least 8-10 distinct content opportunities (not just one).

    For EACH content opportunity, provide these fields in a structured row format:
    - Content Topic/Angle
    - Target Audience Pain Point
    - Current Search/Discussion Volume (High/Medium/Low)
    - Competition Level (Low/Medium/High)
    - Recommended Content Format (Blog, Tutorial, Comparison, Case Study, Video, Guide)
    - Key Questions to Answer (comma-separated list)
    - SEO Keywords (comma-separated list)
    - Unique Angle (one sentence explaining why this matters now)
    - Priority Score (1-10)

    OUTPUT FORMAT:
    Present your findings as a table with clear columns and rows. Each row represents ONE content opportunity. Do NOT number the fields - structure the data as table rows.

    Example format:
    | Content Topic | Pain Point | Volume | Competition | Format | Key Questions | Keywords | Unique Angle | Priority |
    |---------------|------------|---------|-------------|---------|---------------|----------|--------------|----------|
    | [Topic 1] | [Pain] | High | Low | Tutorial | Q1, Q2, Q3 | keyword1, keyword2 | [Angle] | 8 |
    | [Topic 2] | [Pain] | Medium | Medium | Blog | Q1, Q2 | keyword1, keyword2 | [Angle] | 7 |

    After completing your research, create a new Google Spreadsheet titled "Content Opportunities - [Company Name] - [Current Date]" and populate it with your findings in the table format shown above.
    """,
    tools=[
        AgentFactory.create_model_tool(
            model=perplexity_tool.id,
            description="Use Perplexity to research content trends, questions, discussions, and search data"
        ),
        AgentFactory.create_model_tool(
            model=google_sheets_tool.id,
            description="Create and populate Google Spreadsheets with research findings"
        )
    ]
)

print(f"Perplexity Research Agent created! ID: {research_agent.id}")

In [ ]:
# Example usage
response = research_agent.run(
    query="""
    Research content opportunities for:

    Company: aiXplain (aixplain.com)
    Topic Focus: AI agent development, LLM orchestration, and multi-agent systems
    Target Audience: AI engineers, ML developers, and technical decision-makers

    Find 10 high-value content opportunities that will help position aiXplain
    as a thought leader in the AI agent space.
    """
)

print("\n" + "="*50)
print("CONTENT OPPORTUNITIES RESEARCH:")
print("="*50)
print(response.data.output)


In [ ]:
# Follow-up to refine priorities
session_id = response.data.session_id

response = research_agent.run(
    query="Which 3 opportunities should we prioritize first based on quick wins (high volume, low competition)?",
    session_id=session_id
)

print("\n" + "="*50)
print("TOP PRIORITY OPPORTUNITIES:")
print("="*50)
print(response.data.output)

In [ ]:
# Another follow-up for specific niche
response = research_agent.run(
    query="Find 5 more opportunities specifically around RAG (Retrieval-Augmented Generation) use cases",
    session_id=session_id
)

print("\n" + "="*50)
print("RAG-FOCUSED OPPORTUNITIES:")
print("="*50)
print(response.data.output)


In [ ]:
# Deploy the agent
research_agent.deploy()

print(f"\nPerplexity Research Agent deployed!")
print(f"Access at: https://platform.aixplain.com/discover/agent/{research_agent.id}")